In [ ]:
import os
import pygame
import numpy as np
import cv2

from PIL import Image
from rembg import remove

pygame.init()

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
IMAGE_PATH = os.path.join(script_dir, "Resources", "Pizza.png")
REMOVE_BACKGROUND = True

WIDTH, HEIGHT = 1280, 720
FPS = 60

window = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Image Processing Studio")

clock = pygame.time.Clock()

BG = (248, 245, 237)
WHITE = (255, 255, 255)
DARK = (65, 60, 58)
RED = (235, 35, 35)

HEADERS = {
    "Original": (190, 211, 199),
    "Gray Scale": (235, 205, 135),
    "Edges": (229, 173, 150),
    "Contours": (210, 164, 160)
}

title_font = pygame.font.SysFont("Georgia", 25)

CARD_WIDTH = 460
CARD_HEIGHT = 300
HEADER_HEIGHT = 65

cards = [
    (145, 35, "Original"),
    (675, 35, "Gray Scale"),
    (145, 385, "Edges"),
    (675, 385, "Contours")
]


def load_image(path):

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Image Not Found = {path}")

    image = Image.open(path).convert("RGBA")

    if REMOVE_BACKGROUND:
        image = remove(image).convert("RGBA")

    return image

def grayscale_image(image):

    rgba = np.array(image)
    gray = cv2.cvtColor(rgba[:, :, :3],cv2.COLOR_RGB2GRAY)
    result = cv2.cvtColor(gray,cv2.COLOR_GRAY2RGBA)
    result[:, :, 3] = rgba[:, :, 3]

    return Image.fromarray(result)


def edge_detection(image):

    rgba = np.array(image)
    gray = cv2.cvtColor(rgba[:, :, :3],cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray,(5, 5),0)
    edges = cv2.Canny(blurred,60,150)
    result = np.zeros((image.height, image.width, 4),dtype=np.uint8)
    result[edges > 0] = (35, 35, 35, 255)
    
    return Image.fromarray(result)

def contour_detection(image):

    rgba = np.array(image)
    alpha = rgba[:, :, 3]

    if np.any(alpha < 250):
        mask = np.where(alpha > 20,255,0).astype(np.uint8)

    else:
        gray = cv2.cvtColor(rgba[:, :, :3],cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray,(5, 5),0)
        mask = cv2.Canny(blurred,60,150)

    contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = rgba[:, :, :3].copy()

    min_area = max(100,int(image.width * image.height * 0.002))

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h

        if area < min_area:
            continue
        if area > image.width * image.height * 0.95:
            continue

        cv2.rectangle(result,(x, y),(x + w, y + h),(235, 35, 35),5)

    output = np.dstack((result, alpha))
    return Image.fromarray(output)


def create_surface(image):

    return pygame.image.fromstring(
        image.tobytes(),
        image.size,
        image.mode
    )


def fit_image(image, max_width, max_height):

    width, height = image.get_size()

    scale = min(
        max_width / width,
        max_height / height
    )

    new_width = max(1, int(width * scale))
    new_height = max(1, int(height * scale))

    return pygame.transform.smoothscale(
        image,
        (new_width, new_height)
    )

def draw_card(x, y, title, image):

    shadow = pygame.Surface((CARD_WIDTH + 12, CARD_HEIGHT + 12),pygame.SRCALPHA)
    pygame.draw.rect(shadow,(80, 70, 60, 35),(5, 5, CARD_WIDTH, CARD_HEIGHT),border_radius=25)
    window.blit(shadow, (x - 5, y + 2))
    pygame.draw.rect(window,WHITE,(x, y, CARD_WIDTH, CARD_HEIGHT),border_radius=25)
    header_color = HEADERS[title]

    pygame.draw.rect(window,header_color,(x, y, CARD_WIDTH, HEADER_HEIGHT),border_radius=25)
    pygame.draw.rect(window,header_color,(x, y + 32, CARD_WIDTH, HEADER_HEIGHT - 32))

    heading = title_font.render(title,True,DARK)
    window.blit(heading,(x + 28, y + 17))
    preview = fit_image(image,CARD_WIDTH - 45,CARD_HEIGHT - HEADER_HEIGHT - 25)
    preview_rect = preview.get_rect(center=(x + CARD_WIDTH // 2,y + HEADER_HEIGHT + (CARD_HEIGHT - HEADER_HEIGHT) // 2))
    window.blit(preview, preview_rect)

try:

    original_image = load_image(IMAGE_PATH)
    grayscale = grayscale_image(original_image)
    edges = edge_detection(original_image)
    contours = contour_detection(original_image)

except Exception as error:

    print(f"Error = {error}")
    pygame.quit()
    raise SystemExit

images = {
    "Original": create_surface(original_image),
    "Gray Scale": create_surface(grayscale),
    "Edges": create_surface(edges),
    "Contours": create_surface(contours)
}

running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    window.fill(BG)

    for x, y, title in cards:
        draw_card(x,y,title,images[title])

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()